# Chapter 10: Introduction to Spreading in code

Rechecks every number in the chapter (cash-and-carry, the futures calendar spread, the note-over-bond spread, the 3:1 ratio spread, the quote for a three-leg spread, volatility-terms mispricing, the 35% volatility example, the roulette examples) and draws three figures.
Notes: [[chapters/10-introduction-to-spreading]]

Only numpy and matplotlib are needed. Prices and amounts follow the book. Anything marked *our* is not in the book.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from fractions import Fraction

## 1. Cash-and-carry: a spread whose profit does not depend on the price (page 159)
Cash price 700, interest 6% a year, storage and insurance 5 a month, two months. We find the fair forward price, then the profit if the forward actually trades at 725, and check the profit is 8 whatever the commodity is worth at maturity.

In [ ]:
cash, rate, months, storage = 700, 0.06, 2, 5
interest = cash * rate * months / 12
fair_forward = cash + interest + months * storage
print(f"interest {interest:.2f}, storage {months * storage}, fair two-month forward {fair_forward:.2f}  (book 717)")

market_forward = 725
profit = -interest - cash - months * storage + market_forward
print(f"profit if the forward trades at {market_forward}: {profit:.2f}  (book +8)")

# Profit at maturity for any commodity price S: long commodity (bought 700, paid 17 of costs), short forward at 725
S = np.array([600, 700, 800])
long_leg = S - cash - interest - months * storage
short_leg = market_forward - S
print("S      long leg   short leg   spread")
for s, a, b in zip(S, long_leg, short_leg):
    print(f"{s:<6} {a:>9.2f} {b:>11.2f} {a + b:>8.2f}")

**Figure: the legs move, the spread does not.**

In [ ]:
S = np.linspace(600, 800, 101)
long_leg = S - 717
short_leg = 725 - S
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(S, long_leg, label="long commodity (bought 700, costs 17)")
ax.plot(S, short_leg, label="short 2-month forward (sold at 725)")
ax.plot(S, long_leg + short_leg, "k", lw=3, label="the spread: always +8")
ax.axhline(0, color="grey", lw=0.8)
ax.set_xlabel("commodity price at maturity"); ax.set_ylabel("profit or loss")
ax.set_title("Cash-and-carry: two legs, one fixed profit")
ax.legend(fontsize=8); plt.tight_layout()
plt.savefig("../images/ch10-cash-and-carry.png", dpi=130); plt.show()

## 2. A futures calendar spread (pages 160-161)
The four-month forward costs more than the two-month by two extra months of storage and interest, and the interest on the second period is charged on the larger amount (717, not 700).

In [ ]:
f2 = 700 + 700 * 0.06 / 6 + 2 * 5                      # 717.00
interest_2nd = f2 * 0.06 / 6                           # interest on 717 for months 3 and 4
f4 = f2 + interest_2nd + 2 * 5
print(f"2-month forward {f2:.2f}, 4-month forward {f4:.2f}  (book 734.17)")
print(f"fair spread {f4 - f2:.2f}  (book 17.17)")
print(f"if the spread trades at 20 it is {20 - (f4 - f2):.2f} too expensive  (book 2.83)")

# Carry the spread to maturity: buy the 2-month at 717, sell the 4-month at 737
cash_flows = {"borrow 717 for 2 months (interest)": -717 * 0.06 / 6,
              "buy commodity at the 2-month forward price": -717.00,
              "store and insure for 2 months": -10.00,
              "deliver at the 4-month forward price": +737.00}
for k, v in cash_flows.items():
    print(f"{k:<46}{v:>8.2f}")
print(f"{'total':<46}{sum(cash_flows.values()):>8.2f}  (book +2.83)")

## 3. Note-over-bond (NOB) spread, in 32nds (pages 161-162)
Treasury futures are quoted in points and 32nds. We convert to 32nds, subtract, convert back.

In [ ]:
def q(points, thirty_seconds):                      # price as a Fraction of a point
    return Fraction(points) + Fraction(thirty_seconds, 32)

def show(x):                                         # print as 'points n/32'
    pts = int(x); n = (x - pts) * 32
    return f"{pts} {n}/32"

bond, note = q(118, 27), q(116, 14)
sold_at = bond - note
print("opening spread:", show(sold_at), " (book 2 13/32)")

bond2, note2 = q(115, 10), q(113, 7)
bought_back = bond2 - note2
print("closing spread:", show(bought_back), " (book 2 3/32)")
gain = sold_at - bought_back
print("profit from selling the spread and buying it back:", f"{gain * 32} /32", " (book 10/32)")

## 4. A 3:1 ratio spread (page 162)
Commodity B is expected to trade at 3 times Commodity A. We buy 3 A at 120 and sell 1 B at 390, and check that the profit when the ratio is restored is 30 whatever the price level is.

In [ ]:
A0, B0 = 120, 390
print("B / A =", B0 / A0, "(historical ratio 3)")
credit = -3 * A0 + 1 * B0
print("opening credit:", credit, " (book +30)")

for A1 in (50, 100, 200):                           # prices when the 3:1 ratio is restored
    B1 = 3 * A1
    close_cost = 3 * A1 - 1 * B1                    # sell 3 A, buy back 1 B  -> 0
    print(f"A = {A1}, B = {B1}: cost to close {close_cost}, total profit {credit + close_cost}")

# our extra check: what if the ratio does NOT come back?
for B1 in (360, 420):
    A1 = 120
    pnl = credit + (3 * A1 - 1 * B1)
    print(f"A stays 120 and B goes to {B1}: profit {pnl}")

## 5. The crack spread, 3:2:1 (page 163)
Price of the spread = (2 x gasoline) + (1 x heating oil) - (3 x crude oil). The book gives no prices; these are *our* illustrative numbers, all in the same unit.

In [ ]:
gasoline, heating_oil, crude = 90, 95, 80
crack = 2 * gasoline + 1 * heating_oil - 3 * crude
print("3:2:1 crack spread =", crack)

## 6. The market for a three-leg spread (page 164)
Buy A, sell B and C. To buy the spread you buy A at its ask and sell B and C at their bids; to sell it, the reverse.

In [ ]:
quotes = {"A": (128, 131), "B": (47, 49), "C": (68, 70)}       # (bid, ask)
spread_bid = quotes["A"][0] - quotes["B"][1] - quotes["C"][1]   # sell the spread: sell A at its bid, buy B and C at their asks
spread_ask = quotes["A"][1] - quotes["B"][0] - quotes["C"][0]   # buy the spread: buy A at its ask, sell B and C at their bids
print("bid", spread_bid, "ask", spread_ask, "width", spread_ask - spread_bid, " (book 9 and 16)")
print("a market maker might quote 11 / 14: width", 14 - 11)

# any set of leg prices that adds up gives the same spread price
for a, b, c in [(129, 47, 68), (131, 48, 69), (200, 86, 100)]:
    print(f"A {a}, B {b}, C {c}: A - B - C = {a - b - c}")

## 7. Mispricing in volatility terms (page 166)
Two options, theoretical values at 23% volatility, and the implied volatilities of their prices.

In [ ]:
options = [("first", 7.00, 8.00, 26), ("second", 6.00, 6.75, 28)]
print("option   theo   price   over by   implied vol   vol points over 23%")
for name, th, px, iv in options:
    print(f"{name:<8} {th:>5.2f} {px:>6.2f} {px - th:>8.2f} {iv:>10}% {iv - 23:>12}")

## 8. How much margin for error? (pages 166-167)
The trader thinks volatility is 35%, sees a call priced 4.00 with theoretical value 3.50, and sells four calls per underlying contract bought (delta 25). The book says the call is worth 4.50 at 45% volatility.

*Our assumption:* a straight line between the book's two points (3.50 at 35%, 4.50 at 45%), so the call gains 0.10 for every volatility point. That puts the breakeven at 40%.

In [ ]:
edge_per_call = 4.00 - 3.50
for size_calls, size_under in [(4, 1), (40, 10), (400, 100)]:
    print(f"{size_calls} x {size_under}: theoretical edge {size_calls * edge_per_call:.2f}")

vega_pt = (4.50 - 3.50) / 10            # our straight-line assumption: 0.10 per volatility point
def pnl(vol, n_calls):                  # sold at 4.00, call is worth 3.50 + 0.10*(vol-35)
    value = 3.50 + vega_pt * (vol - 35)
    return n_calls * (4.00 - value)

for v in (35, 40, 45):
    print(f"volatility {v}%: P&L of 400 x 100 = {pnl(v, 400):+.0f}")
print("breakeven volatility:", 35 + (4.00 - 3.50) / vega_pt, "%")

In [ ]:
vol = np.linspace(30, 50, 101)
naked = pnl(vol, 400)                                   # breakeven 40
spread = 200 - 20 * (vol - 35)                          # illustration: same +200 edge at 35, breakeven pushed out to 45
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(vol, naked, label="400 x 100 as in the example (breakeven 40%)")
ax.plot(vol, spread, "--", label="same edge after spreading (illustration, breakeven 45%)")
ax.axhline(0, color="grey", lw=0.8)
ax.axvline(35, color="k", ls=":", lw=0.8); ax.text(35.2, -380, "estimate 35%", fontsize=8)
ax.annotate("", xy=(40, 15), xytext=(35, 15), arrowprops=dict(arrowstyle="<->"))
ax.text(35.6, 40, "margin 5 pts", fontsize=8)
ax.annotate("", xy=(45, -35), xytext=(35, -35), arrowprops=dict(arrowstyle="<->"))
ax.text(38.5, -75, "margin 10 pts", fontsize=8)
ax.set_xlabel("volatility that actually happens (%)"); ax.set_ylabel("profit or loss")
ax.set_title("Margin for error = breakeven volatility - my estimate")
ax.legend(fontsize=8, loc="upper right"); plt.tight_layout()
plt.savefig("../images/ch10-margin-for-error.png", dpi=130); plt.show()

## 9. The casino and the roulette table (pages 167-168)
A wheel with 38 numbers pays 35 to 1 (the casino hands back the stake and 35 more, so the "payoff" is 36 per 1 bet). We compare three ways of taking the same $2,000 of bets.

In [ ]:
N = 38
def scenario(bets):                   # bets: list of stakes on different numbers
    total = sum(bets)
    outcomes = [(total - 36 * b, 1 / N) for b in bets]          # a bet wins: casino keeps total, pays 36 x stake
    p_none = (N - len(bets)) / N
    if p_none > 0:
        outcomes.append((total, p_none))                        # no bet wins: casino keeps everything
    mean = sum(x * p for x, p in outcomes)
    sd = (sum(p * (x - mean) ** 2 for x, p in outcomes)) ** 0.5
    worst = min(x for x, p in outcomes)
    best = max(x for x, p in outcomes)
    return outcomes, mean, sd, worst, best

cases = {"one $2,000 bet": [2000], "two $1,000 bets": [1000, 1000], "38 x $1,000 bets": [1000] * 38}
print(f"{'case':<20}{'best':>8}{'worst':>10}{'expected':>10}{'std dev':>10}{'edge on money bet':>20}")
for name, bets in cases.items():
    o, mean, sd, worst, best = scenario(bets)
    print(f"{name:<20}{best:>8.0f}{worst:>10.0f}{mean:>10.2f}{sd:>10.0f}{mean / sum(bets):>19.2%}")

Reading the table: with a $50,000 bankroll the single $2,000 bet can ruin the casino (loss 70,000, chance 1 in 38), while two $1,000 bets cannot (worst loss 34,000). The expected profit per dollar bet is identical (2/38, about 5.26%).

In [ ]:
fig, axs = plt.subplots(1, 3, figsize=(10, 3.4), sharey=True)
for ax, (name, bets) in zip(axs, cases.items()):
    o, mean, sd, worst, best = scenario(bets)
    merged = {}                                         # combine identical outcomes into one bar
    for x, p in o:
        merged[x] = merged.get(x, 0) + p
    xs = sorted(merged); ps = [merged[x] for x in xs]
    ax.bar(range(len(xs)), ps, color=["tab:red" if x < 0 else "tab:green" for x in xs])
    ax.set_xticks(range(len(xs)))
    ax.set_xticklabels([f"{x:+,.0f}" for x in xs], rotation=45, ha="right", fontsize=8)
    ax.set_title(f"{name}\nworst {worst:+,.0f}, std dev {sd:,.0f}", fontsize=9)
axs[0].set_ylabel("probability")
fig.suptitle("Casino profit on each outcome: same edge, very different risk", fontsize=10)
plt.tight_layout(); plt.savefig("../images/ch10-roulette-spreading.png", dpi=130); plt.show()